In [1]:
import pandas as pd
import numpy as np
import os

EXPORT_PATH = r"C:\Users\yipch\OneDrive\Desktop\olist_analytics\exports"
DATA_PATH   = r"C:\Users\yipch\OneDrive\Desktop\olist_analytics\data"

master = pd.read_csv(os.path.join(EXPORT_PATH, "master_orders.csv"),
                     parse_dates=['order_purchase_timestamp',
                                  'order_delivered_customer_date',
                                  'order_estimated_delivery_date'])

order_items = pd.read_csv(os.path.join(DATA_PATH, "olist_order_items_dataset.csv"))
products    = pd.read_csv(os.path.join(DATA_PATH, "olist_products_dataset.csv"))
category    = pd.read_csv(os.path.join(DATA_PATH, "product_category_name_translation.csv"))

print(f"✅ Data loaded: {master.shape}")

✅ Data loaded: (96470, 23)


In [2]:
# Join category names
items_products = order_items.merge(products[['product_id','product_category_name']],
                                    on='product_id', how='left')
items_products = items_products.merge(category, on='product_category_name', how='left')

items_products['category_en'] = items_products['product_category_name_english'].fillna('unknown')

# Revenue per category
cat_revenue = items_products.groupby('category_en')['price'].sum().reset_index()
cat_revenue.columns = ['category', 'revenue']
cat_revenue = cat_revenue.sort_values('revenue', ascending=False).reset_index(drop=True)

# Cumulative %
cat_revenue['revenue_pct']     = cat_revenue['revenue'] / cat_revenue['revenue'].sum() * 100
cat_revenue['cumulative_pct']  = cat_revenue['revenue_pct'].cumsum()

# Flag top 80%
cat_revenue['pareto_flag'] = cat_revenue['cumulative_pct'].apply(
    lambda x: 'Top 80%' if x <= 80 else 'Bottom 20%'
)

top80_count = (cat_revenue['cumulative_pct'] <= 80).sum()
print(f"📊 {top80_count} categories drive 80% of revenue (out of {len(cat_revenue)} total)")
print(cat_revenue.head(10).to_string(index=False))

cat_revenue.to_csv(os.path.join(EXPORT_PATH, "pareto_category.csv"), index=False)
print("\n✅ pareto_category.csv saved!")

📊 17 categories drive 80% of revenue (out of 72 total)
             category    revenue  revenue_pct  cumulative_pct pareto_flag
        health_beauty 1258681.34     9.260700        9.260700     Top 80%
        watches_gifts 1205005.68     8.865783       18.126483     Top 80%
       bed_bath_table 1036988.68     7.629605       25.756088     Top 80%
       sports_leisure  988048.97     7.269533       33.025621     Top 80%
computers_accessories  911954.32     6.709669       39.735290     Top 80%
      furniture_decor  729762.49     5.369200       45.104489     Top 80%
           cool_stuff  635290.85     4.674128       49.778618     Top 80%
           housewares  632248.66     4.651745       54.430363     Top 80%
                 auto  592720.11     4.360916       58.791278     Top 80%
         garden_tools  485256.46     3.570256       62.361534     Top 80%

✅ pareto_category.csv saved!


In [3]:
ltv = master.groupby('customer_unique_id').agg(
    total_orders  = ('order_id', 'nunique'),
    total_revenue = ('total_revenue', 'sum'),
    first_order   = ('order_purchase_timestamp', 'min'),
    last_order    = ('order_purchase_timestamp', 'max')
).reset_index()

# Customer lifespan in days
ltv['lifespan_days'] = (ltv['last_order'] - ltv['first_order']).dt.days

# Average order value per customer
ltv['avg_order_value'] = ltv['total_revenue'] / ltv['total_orders']

# LTV = total revenue (for this dataset — historical LTV)
ltv['ltv'] = ltv['total_revenue']

# LTV segments
ltv['ltv_segment'] = pd.qcut(ltv['ltv'], q=4,
                               labels=['Bronze','Silver','Gold','Platinum'])

print("📊 LTV Segment Summary:")
print(ltv.groupby('ltv_segment').agg(
    customers     = ('customer_unique_id', 'count'),
    avg_ltv       = ('ltv', 'mean'),
    total_revenue = ('ltv', 'sum')
).round(2))

ltv.to_csv(os.path.join(EXPORT_PATH, "customer_ltv.csv"), index=False)
print("\n✅ customer_ltv.csv saved!")

📊 LTV Segment Summary:
             customers  avg_ltv  total_revenue
ltv_segment                                   
Bronze           23340    29.10      679202.21
Silver           23338    65.43     1527015.44
Gold             23334   117.73     2747156.33
Platinum         23338   354.22     8266874.95

✅ customer_ltv.csv saved!


In [4]:
orders_per_cust = master.groupby('customer_unique_id')['order_id'].nunique().reset_index()
orders_per_cust.columns = ['customer_unique_id', 'order_count']

# Distribution
freq_dist = orders_per_cust['order_count'].value_counts().sort_index().reset_index()
freq_dist.columns = ['order_count', 'customers']
freq_dist['customer_pct'] = (freq_dist['customers'] / freq_dist['customers'].sum() * 100).round(2)

print("📊 Order Frequency Distribution:")
print(freq_dist.to_string(index=False))

one_time      = (orders_per_cust['order_count'] == 1).sum()
repeat        = (orders_per_cust['order_count'] > 1).sum()
repeat_rate   = repeat / len(orders_per_cust) * 100

print(f"\n🔁 One-time customers: {one_time:,} ({100-repeat_rate:.1f}%)")
print(f"🔁 Repeat customers:   {repeat:,} ({repeat_rate:.1f}%)")

freq_dist.to_csv(os.path.join(EXPORT_PATH, "order_frequency.csv"), index=False)
print("\n✅ order_frequency.csv saved!")

📊 Order Frequency Distribution:
 order_count  customers  customer_pct
           1      90549         97.00
           2       2573          2.76
           3        181          0.19
           4         28          0.03
           5          9          0.01
           6          5          0.01
           7          3          0.00
           9          1          0.00
          15          1          0.00

🔁 One-time customers: 90,549 (97.0%)
🔁 Repeat customers:   2,801 (3.0%)

✅ order_frequency.csv saved!


In [5]:
payments = pd.read_csv(os.path.join(DATA_PATH, "olist_order_payments_dataset.csv"))

# Payment type mix
pay_type = payments.groupby('payment_type').agg(
    transactions  = ('order_id', 'count'),
    total_value   = ('payment_value', 'sum'),
    avg_value     = ('payment_value', 'mean')
).reset_index().sort_values('total_value', ascending=False)

pay_type['value_pct'] = (pay_type['total_value'] / pay_type['total_value'].sum() * 100).round(2)

print("📊 Payment Type Breakdown:")
print(pay_type.to_string(index=False))

# Installments analysis
install = payments[payments['payment_type'] == 'credit_card'].copy()
install_dist = install['payment_installments'].value_counts().sort_index().reset_index()
install_dist.columns = ['installments', 'count']

print(f"\n💳 Credit Card Installments (top 10):")
print(install_dist.head(10).to_string(index=False))

pay_type.to_csv(os.path.join(EXPORT_PATH, "payment_behavior.csv"), index=False)
install_dist.to_csv(os.path.join(EXPORT_PATH, "installments_dist.csv"), index=False)
print("\n✅ Payment files saved!")

📊 Payment Type Breakdown:
payment_type  transactions  total_value  avg_value  value_pct
 credit_card         76795  12542084.19 163.319021      78.34
      boleto         19784   2869361.27 145.034435      17.92
     voucher          5775    379436.87  65.703354       2.37
  debit_card          1529    217989.79 142.570170       1.36
 not_defined             3         0.00   0.000000       0.00

💳 Credit Card Installments (top 10):
 installments  count
            0      2
            1  25455
            2  12413
            3  10461
            4   7098
            5   5239
            6   3920
            7   1626
            8   4268
            9    644

✅ Payment files saved!


In [6]:
delivery = master[['order_id','customer_state','delivery_days',
                   'delivery_delay_days','review_score']].copy()

# Late vs on-time
delivery['delivery_status'] = delivery['delivery_delay_days'].apply(
    lambda x: 'Late' if x > 0 else 'On Time'
)

# Summary stats
print("📊 Overall Delivery Performance:")
print(delivery[['delivery_days','delivery_delay_days']].describe().round(2))

print(f"\n🚚 On-Time Rate: {(delivery['delivery_status']=='On Time').mean()*100:.1f}%")
print(f"⚠️  Late Rate:    {(delivery['delivery_status']=='Late').mean()*100:.1f}%")

# By state
state_delivery = delivery.groupby('customer_state').agg(
    avg_delivery_days  = ('delivery_days', 'mean'),
    avg_delay_days     = ('delivery_delay_days', 'mean'),
    late_rate_pct      = ('delivery_status', lambda x: (x=='Late').mean()*100),
    avg_review         = ('review_score', 'mean'),
    total_orders       = ('order_id', 'count')
).reset_index().round(2).sort_values('avg_delivery_days', ascending=False)

print(f"\n📊 Delivery by State (Top 10 slowest):")
print(state_delivery.head(10).to_string(index=False))

delivery.to_csv(os.path.join(EXPORT_PATH, "delivery_performance.csv"), index=False)
state_delivery.to_csv(os.path.join(EXPORT_PATH, "delivery_by_state.csv"), index=False)
print("\n✅ Delivery files saved!")

📊 Overall Delivery Performance:
       delivery_days  delivery_delay_days
count       96470.00             96470.00
mean           12.09               -11.88
std             9.55                10.18
min             0.00              -147.00
25%             6.00               -17.00
50%            10.00               -12.00
75%            15.00                -7.00
max           209.00               188.00

🚚 On-Time Rate: 93.2%
⚠️  Late Rate:    6.8%

📊 Delivery by State (Top 10 slowest):
customer_state  avg_delivery_days  avg_delay_days  late_rate_pct  avg_review  total_orders
            RR              28.98          -17.29          12.20        3.90            41
            AP              26.73          -19.69           2.99        4.24            67
            AM              25.99          -19.57           2.76        4.24           145
            AL              24.04           -8.71          21.41        3.86           397
            PA              23.32          -14.07 